# Extra 3 - Treinar um classificador com `nn.Embedding`

Este e o exercicio central para o projeto final: um modelo que **aprende os
embeddings junto com a tarefa**. Pipeline completo:

texto -> limpeza -> vocabulario -> indices com padding -> `nn.Embedding` ->
media com mascara -> `nn.Linear` -> logit -> `BCEWithLogitsLoss`.

Usa `TensorDataset` + `DataLoader` para treinar em batches e uma divisao
treino/validacao. Base: os mesmos SMS spam/ham do modulo 1, criada aqui
mesmo.

Tente resolver antes de olhar o `_solucoes`.

In [1]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)

sms_data = [
    ("ham", "Hey, are we still on for lunch tomorrow?"),
    ("ham", "I'll call you when I get home from work."),
    ("ham", "Can you send me the notes from today's class?"),
    ("ham", "Happy birthday! Hope you have an amazing day."),
    ("ham", "Running a bit late, be there in 10 minutes."),
    ("ham", "Thanks for the ride yesterday, really appreciated."),
    ("ham", "Don't forget the meeting tomorrow morning."),
    ("ham", "Can we reschedule our meeting to tomorrow?"),
    ("ham", "I'll be at the meeting, see you tomorrow."),
    ("ham", "Thanks so much, talk to you tomorrow."),
    ("ham", "Let's grab lunch after the meeting tomorrow."),
    ("ham", "Sorry I missed your call earlier, call me back."),
    ("ham", "Can you call me when you get a chance?"),
    ("ham", "I'll call the doctor to book an appointment."),
    ("ham", "Mom said dinner is ready, come home now."),
    ("ham", "See you at the gym later tonight."),
    ("ham", "Traffic is bad, I might be late for work."),
    ("ham", "Can you pick up the kids from school today?"),
    ("ham", "I forgot my notes at home, can you scan them?"),
    ("ham", "Movie night this weekend? Let me know."),
    ("ham", "Coffee tomorrow morning before work?"),
    ("ham", "Thanks for helping me with the project today."),
    ("ham", "The project meeting got moved to tomorrow."),
    ("ham", "Good night, talk to you tomorrow."),
    ("ham", "Congrats on the new job, so happy for you."),
    ("ham", "Can we do groceries together this weekend?"),
    ("ham", "I'll be home late tonight, don't wait for dinner."),
    ("ham", "Thanks again for everything, means a lot."),
    ("ham", "Let's plan the weekend trip, call me tonight."),
    ("ham", "Dad wants to know if you're coming home tomorrow."),
    ("ham", "Class got cancelled, no notes needed today."),
    ("ham", "I'll bring the notes to the meeting tomorrow."),
    ("ham", "Happy to help, just call me anytime."),
    ("ham", "See you tomorrow at the usual coffee place."),
    ("ham", "The doctor's appointment is confirmed for tomorrow."),
    ("ham", "Sorry for the late reply, was in a meeting."),
    ("ham", "Can you send the project file before tomorrow?"),
    ("ham", "Thanks for lunch, let's do it again soon."),
    ("ham", "I'll pick you up for the gym tomorrow morning."),
    ("ham", "Meeting notes are attached, check before tomorrow."),
    ("ham", "Happy weekend! See you at the gym."),
    ("ham", "Call me back when you're free, nothing urgent."),
    ("ham", "Thanks for the birthday wishes, means a lot."),
    ("ham", "Let's catch up over coffee this weekend."),
    ("ham", "I'm at work, will call you after the meeting."),
    ("ham", "Can you check the notes and call me tonight?"),
    ("ham", "Dinner at mom's tomorrow, don't be late."),
    ("ham", "Thanks for covering my shift today."),
    ("ham", "See you tomorrow, drive safe."),
    ("ham", "The meeting tomorrow is confirmed for 10am."),
    ("ham", "Congrats again, the whole team is proud."),
    ("ham", "Can we push the call to tomorrow afternoon?"),
    ("ham", "I'll send the notes right after the meeting."),
    ("ham", "Thanks for the coffee this morning."),
    ("ham", "Let's meet tomorrow to finish the project."),
    ("ham", "Good luck with the appointment tomorrow."),
    ("ham", "Call me tomorrow, I have some news to share."),
    ("ham", "Thanks for picking up the kids today."),
    ("ham", "See you at the meeting, bring your notes."),
    ("ham", "Home now, dinner will be ready soon."),
    ("ham", "Can't wait for the weekend trip, thanks for planning."),
    ("ham", "Sorry, stuck in traffic, call you when I'm home."),
    ("spam", "URGENT! You have won a free prize, claim now!"),
    ("spam", "Congratulations! You've been selected for a free cash award."),
    ("spam", "WIN a guaranteed cash prize, text WIN to claim now."),
    ("spam", "URGENT! Your mobile number has won a free prize, call now."),
    ("spam", "Free entry to win a cash prize, click the link now."),
    ("spam", "Claim your free cash prize now, urgent reply needed."),
    ("spam", "You have been selected to win a free voucher, claim now."),
    ("spam", "URGENT! Reply now to claim your free cash prize."),
    ("spam", "Winner! You've won a free cash award, text CLAIM now."),
    ("spam", "Free cash prize waiting, call now to claim urgent offer."),
    ("spam", "Exclusive offer: claim your free prize now, urgent!"),
    ("spam", "URGENT! Limited time, claim your free cash now."),
    ("spam", "Congratulations winner! Free cash prize, reply now to claim."),
    ("spam", "Your account has won a free prize, click now to claim."),
    ("spam", "Text WIN now for a chance to claim free cash prize."),
    ("spam", "URGENT offer! Free prize guaranteed, call now to claim."),
    ("spam", "You are a winner! Claim your free cash prize urgent."),
    ("spam", "Free voucher waiting, urgent reply to claim cash prize."),
    ("spam", "Congratulations! Urgent, claim your guaranteed prize now."),
    ("spam", "WIN free cash now, click link, urgent claim required."),
    ("spam", "URGENT! You have a free prize, text CLAIM to collect now."),
    ("spam", "Selected winner, free cash prize, call urgent now."),
    ("spam", "Claim now! Free prize and cash bonus, urgent offer."),
    ("spam", "URGENT! Free cash award waiting, reply CLAIM now."),
    ("spam", "Congratulations, you win! Claim your free prize urgent now."),
    ("ham", "Are you free tonight for dinner?"),
    ("ham", "Congrats on the win, let's celebrate this weekend!"),
    ("ham", "I'll text you the address now."),
    ("ham", "Call me back urgent, mom needs you home."),
    ("ham", "Can I get your number to text you later?"),
    ("ham", "Free tomorrow afternoon? Let's catch up."),
    ("ham", "Great news, call me now, I'm so excited!"),
    ("spam", "Reply now to secure your exclusive reward before it expires."),
    ("spam", "Your account is due a bonus, confirm today to receive it."),
    ("spam", "Act fast, this offer expires today, don't miss out."),
    ("spam", "You are eligible for a special gift, respond immediately."),
    ("spam", "Final notice: your reward is ready, confirm to collect."),
    ("spam", "Selected for an exclusive deal, confirm today to receive."),
]

sms = pd.DataFrame(sms_data, columns=["label", "text"])
print(sms.shape)
sms["label"].value_counts()

(100, 2)

label
ham     69
spam    31
Name: count, dtype: int64

## 3.1 Limpar e rotular

1. `clean_text`: minusculas + regex mantendo so letras e espaco
   (`r"[^a-z\s]"` -> `""`).
2. `y_bin`: `1` para `spam`, `0` para `ham`.

In [2]:
# 3.1
sms["clean_text"] = sms["text"].str.lower().str.replace(r"[^a-z\s]", "", regex=True)
sms["y_bin"] = (sms["label"] == "spam").astype(int)
sms[["clean_text", "y_bin"]].head()

                                    clean_text  y_bin
0       hey are we still on for lunch tomorrow      0
1       ill call you when i get home from work      0
2  can you send me the notes from todays class      0
3  happy birthday hope you have an amazing day      0
4      running a bit late be there in  minutes      0

## 3.2 Vocabulario

1. `tokens` = `clean_text` dividido em palavras.
2. `word2idx` com `<PAD>` = 0 e `<UNK>` = 1; demais palavras a partir de 2.
3. Guarde `vocab_size`.

In [3]:
# 3.2
tokens = sms["clean_text"].str.split()
vocab = sorted({w for msg in tokens for w in msg})
word2idx = {"<PAD>": 0, "<UNK>": 1}
for w in vocab:
    word2idx[w] = len(word2idx)
vocab_size = len(word2idx)
print("vocab_size:", vocab_size)

vocab_size: 225

## 3.3 Codificar com padding

`MAX_LEN = 20`. Funcao `encode(msg)` que recebe a lista de palavras e devolve
uma lista de `MAX_LEN` inteiros:

- cada palavra vira `word2idx.get(w, 1)` (`<UNK>` se nao existir)
- corta em `MAX_LEN`
- completa com `0` (`<PAD>`) ate `MAX_LEN`

Monte `X` (`long`, `(N, MAX_LEN)`) e `y` (`float32`, `(N,)`). Imprima os shapes.

In [4]:
# 3.3
MAX_LEN = 20

def encode(msg):
    idxs = [word2idx.get(w, 1) for w in msg][:MAX_LEN]
    return idxs + [0] * (MAX_LEN - len(idxs))

X = torch.tensor([encode(m) for m in tokens], dtype=torch.long)
y = torch.tensor(sms["y_bin"].values, dtype=torch.float32)
print("X:", X.shape, "| y:", y.shape)

X: torch.Size([100, 20]) | y: torch.Size([100])

## 3.4 Split treino/validacao + DataLoader

1. `train_test_split` com `test_size=0.25`, `stratify=y`, `random_state=0`.
2. `TensorDataset` para cada parte.
3. `DataLoader` de treino com `batch_size=16, shuffle=True`; o de validacao
   com `batch_size=32`.

In [5]:
# 3.4
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0)

train_dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=16, shuffle=True)
val_dl = DataLoader(TensorDataset(X_val, y_val), batch_size=32)
print("treino:", len(X_tr), "| val:", len(X_val))

treino: 75 | val: 25

## 3.5 Modelo: embedding + media com mascara + linear

Classe `MeanEmbeddingClassifier(nn.Module)`:

- `self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)`, `embed_dim = 32`
- `self.fc = nn.Linear(embed_dim, 1)`
- `forward(x)` (`x` = `(batch, MAX_LEN)`):
  1. `v = self.emb(x)` -> `(batch, MAX_LEN, embed_dim)`
  2. `mask = (x != 0).unsqueeze(-1)` -> `(batch, MAX_LEN, 1)`
  3. media so dos tokens validos:
     `(v * mask).sum(1) / mask.sum(1).clamp(min=1)`
  4. `self.fc(...).squeeze(-1)` -> logits `(batch,)`

In [6]:
# 3.5
embed_dim = 32

class MeanEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, x):
        v = self.emb(x)
        mask = (x != 0).unsqueeze(-1)
        media = (v * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.fc(media).squeeze(-1)

model = MeanEmbeddingClassifier(vocab_size, embed_dim)
print(model)

MeanEmbeddingClassifier(
  (emb): Embedding(225, 32, padding_idx=0)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

## 3.6 Loop de treino com avaliacao

- `loss_fn = nn.BCEWithLogitsLoss()`, `optimizer = Adam(lr=0.01)`
- funcao `accuracy(dl)`: poe `model.eval()`, com `torch.no_grad()` percorre o
  loader, converte `torch.sigmoid(logits) > 0.5` e compara com `yb`.
- 40 epocas: em cada uma, `model.train()`, percorre `train_dl`
  (`zero_grad/forward/loss/backward/step`). A cada 10 epocas imprima a loss
  media do treino e a acuracia de treino e validacao.

In [7]:
# 3.6
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

@torch.no_grad()
def accuracy(dl):
    model.eval()
    acertos = total = 0
    for xb, yb in dl:
        pred = (torch.sigmoid(model(xb)) > 0.5).float()
        acertos += (pred == yb).sum().item()
        total += len(yb)
    return acertos / total

for epoch in range(40):
    model.train()
    perdas = []
    for xb, yb in train_dl:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        perdas.append(loss.item())
    if (epoch + 1) % 10 == 0:
        print(f"epoca {epoch + 1:2d} | loss={np.mean(perdas):.4f} "
              f"| acc treino={accuracy(train_dl):.3f} "
              f"| acc val={accuracy(val_dl):.3f}")

epoca 10 | loss=0.0544 | acc treino=1.000 | acc val=0.960
epoca 20 | loss=0.0080 | acc treino=1.000 | acc val=0.960
epoca 30 | loss=0.0037 | acc treino=1.000 | acc val=0.960
epoca 40 | loss=0.0021 | acc treino=1.000 | acc val=0.960

## 3.7 Prever mensagens novas

Funcao `prever(texto)`: limpa igual ao 3.1, tokeniza, `encode`, passa pelo
modelo (`model.eval()`, `no_grad`, `unsqueeze(0)` para virar batch de 1),
aplica `sigmoid` e devolve a probabilidade de spam. Teste com 3 frases.

In [8]:
# 3.7
@torch.no_grad()
def prever(texto):
    model.eval()
    limpo = re.sub(r"[^a-z\s]", "", texto.lower()).split()
    xb = torch.tensor([encode(limpo)], dtype=torch.long)
    return torch.sigmoid(model(xb)).item()

for t in ["claim your free cash prize now",
          "see you at the meeting tomorrow",
          "urgent reply now to win a prize"]:
    print(f"{prever(t):.3f}  {t}")

1.000  claim your free cash prize now
0.000  see you at the meeting tomorrow
0.999  urgent reply now to win a prize

## 3.8 Trocar por embeddings pre-treinados (nota)

Para usar uma matriz ja treinada (skip-gram do modulo 1, CBOW do Extra 2, ou
GloVe/word2vec externos) no lugar da camada aleatoria, bastaria, no
`__init__`:

```python
self.emb = nn.Embedding.from_pretrained(
    torch.tensor(matriz_alinhada, dtype=torch.float32),
    freeze=True,           # True: so treina o classificador; False: fine-tuning
    padding_idx=0,
)
```

O `matriz_alinhada` precisa ter as linhas na **mesma ordem do `word2idx`**
deste notebook — que e exatamente o assunto do Extra 4.

In [9]:
# 3.8 - (celula livre para experimentar)

## Resumo

- A camada `nn.Embedding` aqui e treinada de ponta a ponta com a tarefa.
- `padding_idx=0` + mascara na media = o padding nao suja o vetor da frase
  nem recebe gradiente.
- `TensorDataset`/`DataLoader` = batches e shuffle de graca.
- `BCEWithLogitsLoss` recebe **logit** (sem sigmoid antes).